In [8]:
import torch, numpy as np, time
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torchvision import datasets, transforms
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
import kagglehub
from kagglehub import KaggleDatasetAdapter

import torch.nn as nn
import torch.nn.functional as F


torch.manual_seed(42); np.random.seed(42)
dev = "cuda" if torch.cuda.is_available() else "cpu"

tf = transforms.ToTensor()  # range [0,1]; pakai Sigmoid di output
full = ConcatDataset([datasets.FashionMNIST("data", True, tf, download=True),
                      datasets.FashionMNIST("data", False, tf, download=True)])
n = len(full); n_tr, n_va = int(.8*n), int(.1*n)
tr, va, te = random_split(full, [n_tr, n_va, n-n_tr-n_va],
                          generator=torch.Generator().manual_seed(42))
tr_dl = DataLoader(tr, 128, shuffle=True)
va_dl = DataLoader(va, 128); te_dl = DataLoader(te, 128)

psnr = PeakSignalNoiseRatio(data_range=1.0).to(dev)
ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(dev)

@torch.no_grad()
def eval_recon(model_fn, dl):  # model_fn(x) -> x_hat
    mse = 0; psnr.reset(); ssim.reset(); cnt = 0
    for x, _ in dl:
        x = x.to(dev); xh = model_fn(x)
        mse += ((x-xh)**2).mean().item()*len(x); cnt += len(x)
        psnr.update(xh, x); ssim.update(xh, x)
    return mse/cnt, psnr.compute().item(), ssim.compute().item()

@torch.no_grad()
def latency_ms(fn, runs=50):  # fn() menghasilkan 64 sampel
    for _ in range(5): fn()
    if dev == "cuda": torch.cuda.synchronize()
    t = time.time()
    for _ in range(runs): fn()
    if dev == "cuda": torch.cuda.synchronize()
    return (time.time()-t)/runs*1000

In [9]:
import collections

x, y = next(iter(tr_dl))
print(x.shape, x.min().item(), x.max().item())   # [128,1,28,28], 0.0, 1.0
print(torch.isnan(x).any().item())                # False

print(len(tr), len(va), len(te))                  # 56000, 7000, 7000
cnt = collections.Counter(int(l) for _, l in te)
print(sorted(cnt.items()))                        # tiap kelas ~700, harus seimbang

torch.Size([128, 1, 28, 28]) 0.0 1.0
False
56000 7000 7000
[(0, 700), (1, 659), (2, 708), (3, 714), (4, 707), (5, 660), (6, 685), (7, 706), (8, 738), (9, 723)]


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 10)
    def features(self, x):
        x = x.view(x.size(0),-1)
        h1 = F.relu(self.fc1(x))
        z = F.relu(self.fc2(h1))
        return z
    def forward(self, x):
        return self.fc3(self.features(x))
    
model = MLP().to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

In [12]:
for epoch in range (12):
    model.train()
    for x,y in tr_dl:
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad()
        logits = model(x)
        loss = crit(logits, y)
        loss.backward()
        opt.step()

RuntimeError: addmm: mat1 and mat2 shapes cannot be multiplied (128x784 and 748x256)